# InsertionSort (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_java.ipynb)

Chapter 13 — Correctness. Runnable notebook of `InsertionSort`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the Java code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n \ge 1$ natural numbers. **Output:** the array sorted in non-decreasing order. **Outer invariant:** $a[1..i-1]$ contains the original elements of $a[1..i-1]$ sorted and $a[i..n]$ equals the original. **Inner invariant** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ and these elements are $> key$.

Pseudo-code: section 13.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [2]:
%%writefile InsertionSort.java
public class InsertionSort
{
// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

static boolean equal(int[] x, int xs, int[] y, int ys, int len)     // x[xs..xs+len-1] == y[ys..ys+len-1]
{
    for (int k = 0; k < len; k++)
        if (x[xs + k] != y[ys + k]) return false;
    return true;
}

    // The array with a bar after the sorted part a[1..i-1]
    static String split(int[] a, int i)
    {
        String s = fmt(a, 0, i);
        String t = fmt(a, i, a.length);
        return s.substring(0, s.length() - 1) + " | " + t.substring(1);
    }

    // Outer invariant: a[1..i-1] = original elements of a[1..i-1] sorted, a[i..n] = original
    static void printOuter(int[] a, int[] orig, int i)
    {
        int n = a.length;
        int[] s = java.util.Arrays.copyOf(orig, i);
        java.util.Arrays.sort(s);
        boolean ok = equal(a, 0, s, 0, i) && equal(a, i, orig, i, n - i);
        System.out.println(String.format("%2d | %-26s | %-19s | %s", i + 1, split(a, i), fmt(orig, 0, i), ok ? "ok" : "FALSE"));
    }

    // Inner invariant (I1)-(I3); b = a at the start of the iteration
    static void printInner(int[] a, int[] b, int i, int j, int key)
    {
        int ip = i + 1, jp = j + 1;                // indices of the pseudo-code
        boolean ok = equal(a, 0, b, 0, jp) && equal(a, jp + 1, b, jp, ip - jp - 1);
        for (int k = jp + 1; k < ip; k++)
            if (a[k] <= key) ok = false;
        String cond = "j = 0: false";
        if (j >= 0)
        {
            boolean t = a[j] > key;
            cond = "a[" + jp + "] = " + a[j] + (t ? " > " : " <= ") + key + ": " + (t ? "true" : "false");
        }
        System.out.println(String.format("%d | %-15s | %-12s | %-9s | %-20s | %s", jp, fmt(a, 0, a.length), fmt(a, 0, jp), fmt(a, jp + 1, ip), cond, ok ? "ok" : "FALSE"));
    }

    // InsertionSort(a, n)
    static void insertionSort(int[] a, boolean showOuter, int traceInnerAt)
    {
        int n = a.length;
        int[] orig = a.clone();                    // copy of the input: it is only needed to check the invariant
        int i = 1;                                 // i := 2
        if (showOuter) printOuter(a, orig, i);
        while (i < n)                              // i <= n
        {
            int[] b = a.clone();                   // content of a at the start of the iteration (only for the inner invariant)
            int key = a[i];
            int j = i - 1;
            boolean tr = (i + 1 == traceInnerAt);
            if (tr) printInner(a, b, i, j, key);
            while (j >= 0 && a[j] > key)           // j >= 1 and a[j] > key
            {
                a[j + 1] = a[j];
                j = j - 1;
                if (tr) printInner(a, b, i, j, key);
            }
            a[j + 1] = key;
            i = i + 1;
            if (showOuter) printOuter(a, orig, i);
        }
    }

    public static void main(String[] args)
    {
        // The outer loop: a[1..5] = (5, 2, 4, 1, 3), example of the lecture notes
        int[] a = {5, 2, 4, 1, 3};
        int n = a.length;
        System.out.printf("a = %s\n", fmt(a, 0, n));
        System.out.printf(" i | a (sorted | rest)          | original a[1..i-1]  | invariant\n");
        insertionSort(a, true, 0);
        System.out.printf("sorted: %s\n", fmt(a, 0, n));

        // The inner loop at iteration i = 5, with key = 3 (a = (1, 2, 4, 5, 3) at the start)
        System.out.printf("\n");
        int[] c = {5, 2, 4, 1, 3};
        System.out.printf("j | a               | a[1..j]      | a[j+2..i] | condition            | invariant\n");
        insertionSort(c, false, 5);
        System.out.printf("after line 10 (a[j+1] = key): a = %s\n", fmt(c, 0, n));
    }
}

Writing InsertionSort.java


In [3]:
!javac InsertionSort.java && java InsertionSort

a = [5, 2, 4, 1, 3]
 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
sorted: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condition            | invariant
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
after line 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Try it yourself.** Change the array: every row of the outer table must stay `ok`.

In [4]:
%%writefile ProvaInsertionSort.java
public class ProvaInsertionSort
{
    public static void main(String[] args)
    {
        int[] a = {6, 3, 8, 1, 9, 2};      // <-- change the array
        System.out.printf(" i | a (sorted | rest)          | original a[1..i-1]  | invariant\n");
        InsertionSort.insertionSort(a, true, 0);
    }
}

Writing ProvaInsertionSort.java


In [5]:
!javac ProvaInsertionSort.java && java ProvaInsertionSort

 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).